In [1]:
SPARK_CONNECT_HOST = "localhost" # if running via an SSH tunnel
#SPARK_CONNECT_HOST = "spark-connector-0-driver-svc" # for running in the Kubernetes cluster

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("test").remote(f"sc://{SPARK_CONNECT_HOST}:15002").getOrCreate()
print("session created")

session created


Take patient data from AWS and bed occupancy data from GCP and join them

Before running the next line, you need to register the warehouses and the tables in Polaris.
See instructions in the docs at [README.md](https://github.com/emrys-health/afon-core/blob/main/afon-infra/README.md).

In [4]:
NAMESPACE = "samples2"
patients = spark.read.table(f"aws.{NAMESPACE}.patients")
occupancy = spark.read.table(f"gcp.{NAMESPACE}.occupancy")
data = patients.join(occupancy, patients.id == occupancy.id)
data.show()

+---+---+---+-------+
| id|age| id|numDays|
+---+---+---+-------+
|  1|  1|  1|      0|
|  2|  2|  2|      1|
|  3|  3|  3|      0|
|  4|  4|  4|      3|
|  5|  5|  5|      5|
|  6|  6|  6|      6|
|  7|  7|  7|      4|
|  8|  8|  8|      8|
|  9|  9|  9|      7|
| 10| 10| 10|      6|
| 11| 11| 11|      2|
| 12| 12| 12|      8|
| 13| 13| 13|      0|
| 14| 14| 14|      9|
| 15| 15| 15|      5|
| 16| 16| 16|      4|
| 17| 17| 17|     16|
| 18| 18| 18|     11|
| 19| 19| 19|     18|
| 20| 20| 20|     18|
+---+---+---+-------+
only showing top 20 rows


In [14]:
from pyspark.ml.feature import Interaction, VectorAssembler
from pyspark.ml.stat import Correlation

featuresColName = "features"
assembler = (VectorAssembler()
        .setInputCols(["age", "numDays"])
        .setOutputCol(featuresColName))
df = assembler.transform(data)

r1 = Correlation.corr(df, "features").head()
print("Pearson correlation matrix:\n" + str(r1[0]))

Pearson correlation matrix:
DenseMatrix([[1.        , 0.64679249],
             [0.64679249, 1.        ]])
